In [ ]:
import numpy as np
import os
from os import walk
import datetime
import collections
from os.path import exists, getsize, join
import pandas as pd

print("✓ Libraries imported successfully")

import torch
print("CUDA available:", torch.cuda.is_available())

ok = hasattr(torch, "version") and getattr(torch.version, "cuda", None) is not None
print("Torch CUDA build info ok:", ok)

if torch.cuda.is_available() and ok:
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("Skip GPU name check due to torch build issue")


# Cell 2: Configuration
# Cấu hình đường dẫn
INPUT_TRAFFIC_PATH = '/kaggle/input/datasets/linhhphng/mac-new-50labels500/50labels500_mac_new'  # Folder chứa tất cả file traffic
OUTPUT_PATH = '/kaggle/working/output/'  # Đường dẫn output

# Tham số
# ============================================================================
# CONFIGURATION
# ============================================================================
MAX_PACKETS = 10000
TRAIN_SAMPLES_PER_LABEL = 50   # số samples train mỗi label
TEST_SAMPLES_PER_LABEL  = 100    # số samples test mỗi label
MIN_FILES_PER_LABEL     = 50
MAX_LABELS = 100

print(f"Input path: {INPUT_TRAFFIC_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Max labels to extract: {MAX_LABELS}")

In [ ]:
# Cell 3: Extract Labels from Filenames
def extract_label_from_filename(filename):
    """
    Extract label từ tên file
    Format: label_name_number.txt
    VD: traffic_app.com.brochili_40.txt -> label = app.com.brochili
    """
    # Bỏ extension .txt
    name = filename.replace('.txt', '')
    
    # Tìm vị trí số cuối cùng (sau dấu _ cuối)
    # Split theo _ và lấy phần trước số cuối
    parts = name.split('_')
    
    # Tìm index của phần tử cuối cùng là số
    label_parts = []
    for i, part in enumerate(parts):
        if part.isdigit():
            # Nếu gặp số, lấy tất cả phần trước đó làm label
            label_parts = parts[:i]
            break
    else:
        # Nếu không có số, lấy tất cả
        label_parts = parts
    
    # Nếu có prefix "traffic_" thì bỏ đi
    label = '_'.join(label_parts)
    if label.startswith('traffic_'):
        label = label[8:]  # Bỏ "traffic_"
    
    return label

print("✓ extract_label_from_filename() defined")

# Cell 4: Scan All Traffic Files and Map Labels
def scan_traffic_files(input_path):
    """Scan tất cả files và extract unique labels, map sang số"""
    
    if not os.path.exists(input_path):
        print(f"❌ Path not found: {input_path}")
        return {}, {}
    
    # Lấy tất cả file .txt
    all_files = [f for f in os.listdir(input_path) if f.endswith('.txt')]
    
    print(f"\n📁 Found {len(all_files)} traffic files")
    
    # Extract labels từ tất cả files
    label_set = set()
    file_label_map = {}  # Map filename -> label_name
    
    for filename in all_files:
        label = extract_label_from_filename(filename)
        label_set.add(label)
        file_label_map[filename] = label
    
    # Sort labels và chỉ lấy MAX_LABELS đầu tiên
    sorted_labels = sorted(list(label_set))
    
    # Giới hạn số labels
    if len(sorted_labels) > MAX_LABELS:
        print(f"\n⚠️  Total labels found: {len(sorted_labels)}")
        print(f"⚠️  Limiting to first {MAX_LABELS} labels")
        sorted_labels = sorted_labels[:MAX_LABELS]
    
    label_mapping = {label: idx for idx, label in enumerate(sorted_labels)}
    
    # Lọc file_label_map chỉ giữ labels trong mapping
    filtered_file_label_map = {
        filename: label 
        for filename, label in file_label_map.items() 
        if label in label_mapping
    }
    
    print(f'\n🏷️  Using {len(sorted_labels)} labels:')
    for label, idx in label_mapping.items():
        # Đếm số files của label này
        count = sum(1 for f, l in filtered_file_label_map.items() if l == label)
        print(f'  {idx}: {label} ({count} files)')
    
    total_files = sum(1 for _ in filtered_file_label_map.values())
    print(f'\n✓ Total files to process: {total_files}')
    
    return label_mapping, filtered_file_label_map
    
# # Cell 4: Scan All Traffic Files and Map Labels (folder con = label)
# def scan_traffic_files(input_path):
#     """
#     Cấu trúc: input_path/label_name/*.txt
#     Mỗi folder con là 1 label, chứa các file sample của label đó.
#     """
#     if not os.path.exists(input_path):
#         print(f"❌ Path not found: {input_path}")
#         return {}, {}

#     # Lấy tất cả folder con (mỗi folder = 1 label)
#     all_labels = [d for d in os.listdir(input_path)
#                   if os.path.isdir(join(input_path, d))]

#     print(f"\n📁 Found {len(all_labels)} label folders")

#     sorted_labels = sorted(all_labels)

#     if len(sorted_labels) > MAX_LABELS:
#         print(f"\n⚠️  Total labels found: {len(sorted_labels)}")
#         print(f"⚠️  Limiting to first {MAX_LABELS} labels")
#         sorted_labels = sorted_labels[:MAX_LABELS]

#     label_mapping = {label: idx for idx, label in enumerate(sorted_labels)}

#     # file_label_map: key = "label_name/filename.txt" (đường dẫn tương đối), value = label_name
#     file_label_map = {}
#     for label in sorted_labels:
#         label_dir = join(input_path, label)
#         files_in_label = [f for f in os.listdir(label_dir) if f.endswith('.txt')]
#         for fname in files_in_label:
#             rel_path = join(label, fname)  # relative path: label_name/filename.txt
#             file_label_map[rel_path] = label

#     print(f'\n🏷️  Using {len(sorted_labels)} labels:')
#     for label, idx in label_mapping.items():
#         count = sum(1 for f, l in file_label_map.items() if l == label)
#         print(f'  {idx}: {label} ({count} files)')

#     total_files = len(file_label_map)
#     print(f'\n✓ Total files to process: {total_files}')

#     return label_mapping, file_label_map


print("✓ scan_traffic_files() defined")


In [ ]:
# Cell 5: Helper Functions - Get Local IP
def get_local_ip(path):
    """Xác định IP local dựa trên tần suất xuất hiện"""
    ip_list = []
    with open(path, 'r') as file:
        packets = file.readlines()

    for packet_line in packets:
        packet = packet_line.strip()
        if not packet:
            continue
        strs = packet.split(',')
        if len(strs) < 6:
            continue
        timestamp, src_ip, sport, dst_ip, dport, packet_size = strs[:6]
        ip_list.append(src_ip)
        ip_list.append(dst_ip)

    counter = collections.Counter(ip_list)
    if len(counter) == 0:
        return -1
    local_ip = counter.most_common(1)[0][0]
    return local_ip

print("✓ get_local_ip() defined")

In [ ]:
# from scipy import stats
# import numpy as np
# from sklearn.model_selection import train_test_split
# from scipy import stats
# import numpy as np

# # Cell 7: Parse Traffic File (File-based)
# def parse_pcap(path):
#     """Parse file traffic và extract tất cả packets"""
#     local_ip = get_local_ip(path)
#     all_packets = []
    
#     if local_ip == -1:
#         return all_packets

#     with open(path, 'r') as file:
#         packets = file.readlines()

#     for packet_line in packets:
#         packet = packet_line.strip()
#         if not packet:
#             continue
#         strs = packet.split(',')
#         if len(strs) < 6:
#             continue
            
#         timestamp, src_ip, sport, dst_ip, dport, packet_size = strs[:6]
#         arrival_time = datetime.datetime.fromtimestamp(float(timestamp))
        
#         sport = int(sport)
#         dport = int(dport)
#         length = int(packet_size)
        
#         # Xác định direction: 0 = outgoing, 1 = incoming
#         if src_ip == local_ip:
#             direction = 0  # outgoing
#         else:
#             direction = 1  # incoming
        
#         all_packets.append([arrival_time, direction, length])
    
#     return all_packets

# print("✓ parse_pcap() defined")


# # Cell 8 — THAY THẾ find_max_packet_count() bằng scan đơn giản hơn
# def scan_valid_files(input_traffic_path, file_label_map):
#     """
#     Scan files, chỉ bỏ file rỗng / lỗi.
#     Không filter theo số packets nữa — cắt hoặc pad ở bước generate.
#     """
#     print("\n🔍 Scanning files (skip empty/error only)...")

#     valid_files = set()
#     stats = {'valid': 0, 'empty': 0, 'error': 0}
#     total = len(file_label_map)

#     for i, filename in enumerate(file_label_map.keys(), 1):
#         path = join(input_traffic_path, filename)
#         if not exists(path) or getsize(path) == 0:
#             stats['empty'] += 1
#             continue
#         try:
#             # chỉ đọc dòng đầu để kiểm tra format
#             with open(path, 'r') as f:
#                 first = f.readline().strip()
#             if first and len(first.split(',')) >= 6:
#                 valid_files.add(filename)
#                 stats['valid'] += 1
#             else:
#                 stats['error'] += 1
#         except Exception:
#             stats['error'] += 1

#         if i % 500 == 0:
#             print(f"  Progress: {i}/{total} scanned... (valid: {stats['valid']})")

#     print(f"\n📊 Scan results:")
#     print(f"  ✓ Valid files : {stats['valid']}")
#     print(f"  ⚠ Empty/miss  : {stats['empty']}")
#     print(f"  ❌ Format err  : {stats['error']}")
#     return valid_files

# print("✓ scan_valid_files() defined")

# # Cell 9 — THAY THẾ generate_raw_sample()
# def generate_raw_sample(traffic_file_path, label_id):
#     """
#     Parse file và tạo sample có đúng MAX_PACKETS packets.
#     - Ít hơn MAX_PACKETS → padding 0
#     - Nhiều hơn MAX_PACKETS → cắt lấy MAX_PACKETS đầu
#     - File rỗng / lỗi → trả về None
#     """
#     if not exists(traffic_file_path) or getsize(traffic_file_path) == 0:
#         return None

#     try:
#         all_packets = parse_pcap(traffic_file_path)
#         if len(all_packets) == 0:
#             return None

#         start_time = all_packets[0][0]
#         raw_data = np.zeros((MAX_PACKETS, 4), dtype=np.float32)

#         # Cắt nếu dài hơn, pad nếu ngắn hơn
#         n = min(len(all_packets), MAX_PACKETS)
#         for i in range(n):
#             pkt = all_packets[i]
#             rel_time = (pkt[0] - start_time).total_seconds()
#             raw_data[i] = [label_id, rel_time, pkt[1], pkt[2]]

#         return raw_data.flatten().tolist()

#     except Exception as e:
#         print(f'  ❌ Error {traffic_file_path}: {e}')
#         return None

# print("✓ generate_raw_sample() defined")

# def process_traffic_data(input_traffic_path, output_path,
#                          train_per_label=TRAIN_SAMPLES_PER_LABEL,
#                          test_per_label=TEST_SAMPLES_PER_LABEL,
#                          random_seed=42):
#     """
#     Pipeline:
#     - Lấy TẤT CẢ files hợp lệ (không filter theo độ dài)
#     - Cắt / pad về đúng MAX_PACKETS
#     - Mỗi label: lấy đúng train_per_label train + test_per_label test
#       (nếu không đủ file thì lấy tối đa có thể, in warning)
#     """
#     need_per_label = train_per_label + test_per_label

#     print("="*60)
#     print("🚀 STARTING DATA EXTRACTION")
#     print(f"   MAX_PACKETS       = {MAX_PACKETS}")
#     print(f"   TRAIN per label   = {train_per_label}")
#     print(f"   TEST  per label   = {test_per_label}")
#     print(f"   Need  per label   = {need_per_label}")
#     print("="*60)

#     os.makedirs(output_path, exist_ok=True)
#     emb_out  = join(output_path, 'embedding_data')
#     test_out = join(output_path, 'test_data')
#     os.makedirs(emb_out,  exist_ok=True)
#     os.makedirs(test_out, exist_ok=True)

#     # --- Scan labels ---
#     label_mapping, file_label_map = scan_traffic_files(input_traffic_path)
#     if not label_mapping:
#         print("❌ No labels found!"); return

#     # --- Scan valid files ---
#     valid_files = scan_valid_files(input_traffic_path, file_label_map)
#     if not valid_files:
#         print("❌ No valid files!"); return

#     # --- Group by label ---
#     files_by_label = {}
#     for fname, lname in file_label_map.items():
#         if fname not in valid_files:
#             continue
#         files_by_label.setdefault(lname, []).append(fname)

#     # Bỏ label quá ít file
#     filtered_labels, removed = {}, {}
#     for lname, files in files_by_label.items():
#         if len(files) >= MIN_FILES_PER_LABEL:
#             filtered_labels[lname] = files
#         else:
#             removed[lname] = len(files)

#     print(f"\n📊 Label filtering (min {MIN_FILES_PER_LABEL} files):")
#     print(f"  ✓ Keep: {len(filtered_labels)}   ❌ Drop: {len(removed)}")
#     if removed:
#         for lname, cnt in sorted(removed.items()):
#             print(f"    - {lname}: {cnt} files")

#     if not filtered_labels:
#         print("❌ No labels passed the filter!"); return

#     # Rebuild label mapping
#     new_mapping = {lname: i for i, lname in enumerate(sorted(filtered_labels))}
#     label_df = pd.DataFrame(list(new_mapping.items()), columns=['label_name','label_id'])
#     label_df.to_csv(join(emb_out,  'label_mapping.csv'), index=False)
#     label_df.to_csv(join(test_out, 'label_mapping.csv'), index=False)

#     print(f"\n📏 Fixed sequence length: {MAX_PACKETS} packets × 4 cols = {MAX_PACKETS*4} features")

#     # ✅ Mở file CSV để ghi incremental — không tích lũy vào list
#     emb_fpath  = join(emb_out,  'raw_packet_data.csv')
#     test_fpath = join(test_out, 'raw_packet_data.csv')

#     emb_file  = open(emb_fpath,  'w', newline='')
#     test_file = open(test_fpath, 'w', newline='')
#     import csv
#     emb_writer  = csv.writer(emb_file)
#     test_writer = csv.writer(test_file)

#     split_stats = []
#     emb_count = 0
#     test_count = 0
#     np.random.seed(random_seed)

#     try:
#         for lname in sorted(filtered_labels):
#             lid   = new_mapping[lname]
#             files = sorted(filtered_labels[lname])
#             shuffled = list(np.random.permutation(files))

#             available    = len(shuffled)
#             actual_train = train_per_label
#             actual_test  = test_per_label

#             if available < need_per_label:
#                 ratio        = train_per_label / need_per_label
#                 actual_train = max(1, int(available * ratio))
#                 actual_test  = max(0, available - actual_train)
#                 print(f'\n⚠️  {lname}: only {available} files '
#                       f'(need {need_per_label}) → use {actual_train} train / {actual_test} test')
#             else:
#                 print(f'\n📂 {lname} (id={lid}): {available} files → '
#                       f'{actual_train} train / {actual_test} test')

#             emb_files  = shuffled[:actual_train]
#             test_files = shuffled[actual_train : actual_train + actual_test]

#             def _process(flist, writer):
#                 count = 0
#                 for fname in flist:
#                     sample = generate_raw_sample(join(input_traffic_path, fname), lid)
#                     if sample is not None:
#                         writer.writerow(sample)  # ✅ ghi thẳng, không giữ trong RAM
#                         count += 1
#                 return count

#             ec = _process(emb_files,  emb_writer)
#             tc = _process(test_files, test_writer)

#             # ✅ Flush sau mỗi label để đảm bảo ghi xuống disk
#             emb_file.flush()
#             test_file.flush()

#             emb_count  += ec
#             test_count += tc
#             print(f'  ✓ generated: {ec} train, {tc} test')

#             split_stats.append({
#                 'label_name'   : lname,
#                 'label_id'     : lid,
#                 'total_files'  : available,
#                 'train_samples': ec,
#                 'test_samples' : tc,
#             })

#     finally:
#         emb_file.close()
#         test_file.close()

#     # In kết quả
#     emb_sz  = os.path.getsize(emb_fpath)  / 1024 / 1024
#     test_sz = os.path.getsize(test_fpath) / 1024 / 1024
#     print(f'\n{"="*60}')
#     print(f'✅ TRAIN (embedding): {emb_count:,} samples  ({emb_sz:.1f} MB)  → {emb_fpath}')
#     print(f'✅ TEST             : {test_count:,} samples  ({test_sz:.1f} MB)  → {test_fpath}')

#     stats_df = pd.DataFrame(split_stats)
#     stats_df.to_csv(join(output_path, 'split_statistics.csv'), index=False)

#     print(f'\n{"="*60}')
#     print('✅ DONE')
#     print(f'   Labels : {len(filtered_labels)}')
#     print(f'   Train  : {emb_count:,}')
#     print(f'   Test   : {test_count:,}')
#     print(f'{"="*60}')
#     print(stats_df.to_string(index=False))
    
# print("✓ process_traffic_data() defined")

# # Cell 11 — gọi hàm
# process_traffic_data(
#     INPUT_TRAFFIC_PATH,
#     OUTPUT_PATH,
#     train_per_label=TRAIN_SAMPLES_PER_LABEL,   # ← đổi số này tùy ý
#     test_per_label=TEST_SAMPLES_PER_LABEL,     # ← đổi số này tùy ý
#     random_seed=42,
# )

In [ ]:
"""
Author: Yonglong Tian (yonglong@mit.edu)
Date: May 07, 2020
"""
from __future__ import print_function

import torch
import torch.nn as nn
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print(nn.Linear(10, 5))


class SupConLoss(nn.Module):
    """Supervised Contrastive Learning: https://arxiv.org/pdf/2004.11362.pdf.
    It also supports the unsupervised contrastive loss in SimCLR"""
    def __init__(self, temperature=0.1, contrast_mode='all',
                 base_temperature=0.1):
        super(SupConLoss, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):
        """Compute loss for model. If both `labels` and `mask` are None,
        it degenerates to SimCLR unsupervised loss:
        https://arxiv.org/pdf/2002.05709.pdf

        Args:
            features: hidden vector of shape [bsz, n_views, ...].
            labels: ground truth of shape [bsz].
            mask: contrastive mask of shape [bsz, bsz], mask_{i,j}=1 if sample j
                has the same class as sample i. Can be asymmetric.
        Returns:
            A loss scalar.
        """
        device = (torch.device('cuda')
                  if features.is_cuda
                  else torch.device('cpu'))

        if len(features.shape) < 3:
            raise ValueError('`features` needs to be [bsz, n_views, ...],'
                             'at least 3 dimensions are required')
        if len(features.shape) > 3:
            features = features.view(features.shape[0], features.shape[1], -1)

        batch_size = features.shape[0]
        if labels is not None and mask is not None:
            raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None:
            mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        elif labels is not None:
            labels = labels.contiguous().view(-1, 1)
            if labels.shape[0] != batch_size:
                raise ValueError('Num of labels does not match num of features')
            mask = torch.eq(labels, labels.T).float().to(device)
        else:
            mask = mask.float().to(device)

        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one':
            anchor_feature = features[:, 0]
            anchor_count = 1
        elif self.contrast_mode == 'all':
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else:
            raise ValueError('Unknown mode: {}'.format(self.contrast_mode))

        # compute logits
        anchor_dot_contrast = torch.div(
            torch.matmul(anchor_feature, contrast_feature.T),
            self.temperature)
        # for numerical stability
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # tile mask
        mask = mask.repeat(anchor_count, contrast_count)
        # mask-out self-contrast cases
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(batch_size * anchor_count).view(-1, 1).to(device),
            0
        )
        mask = mask * logits_mask

        # compute log_prob
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))

        # compute mean of log-likelihood over positive
        # modified to handle edge cases when there is no positive pair
        # for an anchor point. 
        # Edge case e.g.:- 
        # features of shape: [4,1,...]
        # labels:            [0,1,1,2]
        # loss before mean:  [nan, ..., ..., nan] 
        mask_pos_pairs = mask.sum(1)
        mask_pos_pairs = torch.where(mask_pos_pairs < 1e-6, 1, mask_pos_pairs)
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs

        # loss
        loss = - (self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss

In [ ]:
import numpy as np
import pandas as pd
import os
import torch
from torch import nn
import torch.nn.functional as F
import random
import math
import time
import sys
import datetime

# ============================================================================
# CONFIGURATION
# ============================================================================
IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    RAW_DATA_PATH = '/kaggle/input/datasets/linhhphng/100-topsearch/100-topsearch/icloud_100/train_data.csv'
    LABEL_MAPPING_PATH = '/kaggle/input/datasets/linhhphng/100-topsearch/100-topsearch/icloud_100/label_mapping.csv'
    MODEL_PATH = '/kaggle/working/pretrain_topsearch10.pth'

# Training parameters
BATCH_SIZE = 32
LEARNING_RATE = 0.1
WEIGHT_DECAY = 1e-4
MOMENTUM = 0.9
EPOCHS = 200
TEMPERATURE = 0.1

# Learning rate schedule
LR_DECAY_EPOCHS = [700, 800, 900]
LR_DECAY_RATE = 0.1
COSINE_SCHEDULE = True
WARMUP = True
WARMUP_EPOCHS = 10
WARMUP_FROM = 0.01

# Model parameters
HIDDEN_SIZE = 256
EMBEDDING_SIZE = 128


print(f"Running on: {'KAGGLE' if IS_KAGGLE else 'LOCAL'}")
print(f"Raw data path: {RAW_DATA_PATH}")

# ============================================================================
# UTILITIES (from original code)
# ============================================================================

class AverageMeter:
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def adjust_learning_rate(optimizer, epoch):
    """Decay the learning rate based on schedule (following original code)"""
    lr = LEARNING_RATE
    
    if COSINE_SCHEDULE:
        eta_min = lr * (LR_DECAY_RATE ** 3)
        lr = eta_min + (lr - eta_min) * (1 + math.cos(math.pi * epoch / EPOCHS)) / 2
    else:
        steps = np.sum(epoch > np.asarray(LR_DECAY_EPOCHS))
        if steps > 0:
            lr = lr * (LR_DECAY_RATE ** steps)
    
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    
    return lr


def warmup_learning_rate(optimizer, epoch, batch_id, total_batches):
    """Warmup learning rate (following original code)"""
    if not WARMUP or epoch > WARMUP_EPOCHS:
        return
    
    p = (batch_id + (epoch - 1) * total_batches) / (WARMUP_EPOCHS * total_batches)
    
    if COSINE_SCHEDULE:
        eta_min = LEARNING_RATE * (LR_DECAY_RATE ** 3)
        warmup_to = eta_min + (LEARNING_RATE - eta_min) * (
                1 + math.cos(math.pi * WARMUP_EPOCHS / EPOCHS)) / 2
    else:
        warmup_to = LEARNING_RATE
    
    lr = WARMUP_FROM + p * (warmup_to - WARMUP_FROM)
    
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr


def save_model(model, optimizer, epoch, save_path):
    """Save model checkpoint"""
    print(f'💾 Saving model to {save_path}')
    state = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch,
        'model_config': {
            'max_packets': model.max_packets if hasattr(model, 'max_packets') else MAX_PACKETS,
            'hidden_size': HIDDEN_SIZE,
            'embedding_size': EMBEDDING_SIZE,
        }
    }
    torch.save(state, save_path)


# ============================================================================
# DATA PROCESSING
# ============================================================================

def load_and_process_raw_data(raw_data_path, label_mapping_path):
    """Load raw packet data and reshape to (N, max_packets, 3)"""
    print("\n" + "="*60)
    print("LOADING RAW PACKET DATA")
    print("="*60)
    
    # Load label mapping
    if os.path.exists(label_mapping_path):
        label_df = pd.read_csv(label_mapping_path)
        label_names = dict(zip(label_df['label_name'], label_df['label_id']))
        print(f"\n✓ Loaded {len(label_names)} label mappings")
    else:
        print("⚠️  Label mapping not found")
        label_names = {}
    
    # Load raw data
    print(f"\n📂 Loading raw data from: {raw_data_path}")
    raw_df = pd.read_csv(raw_data_path, header=None)
    
    print(f"✓ Loaded {len(raw_df)} samples")
    print(f"✓ Total columns: {len(raw_df.columns)}")
    
    # Determine max_packets from shape
    total_cols = len(raw_df.columns)
    max_packets = total_cols // 4
    
    print(f"✓ Detected shape: ({max_packets}, 4) per sample")
    
    # Reshape data
    print(f"\n🔄 Reshaping data...")
    all_samples = []
    all_labels = []
    
    for idx, row in raw_df.iterrows():
        sample = row.values.reshape(max_packets, 4)
        label = int(sample[0, 0])
        sample_features = sample[:, 1:]  # Keep [time, direction, size]
        
        all_samples.append(sample_features)
        all_labels.append(label)
        
        if (idx + 1) % 100 == 0:
            print(f"  Processed {idx + 1}/{len(raw_df)} samples...")
    
    X = np.array(all_samples, dtype=np.float32)
    y = np.array(all_labels, dtype=np.int64)
    
    print(f"\n✅ Data reshaping complete:")
    print(f"  X shape: {X.shape}")
    print(f"  y shape: {y.shape}")
    print(f"  Unique labels: {len(np.unique(y))}")
    
    # Label distribution
    unique_labels, counts = np.unique(y, return_counts=True)
    print(f"\n📊 Label distribution:")
    for label, count in zip(unique_labels, counts):
        label_name = label_names.get(label, f"Label_{label}")
        print(f"  {label_name} (ID={label}): {count} samples")
    
    return X, y, label_names, max_packets


def organize_data_by_label(X, y, samples_per_class=250):
    """Organize data by label for contrastive learning - lấy tối đa samples_per_class cho mỗi class"""
    print("\n📦 Organizing data by label...")
    
    data_table = {}
    for sample, label in zip(X, y):
        if label not in data_table:
            data_table[label] = []
        data_table[label].append(sample)
    
    # Filter labels with multiple samples và giới hạn số lượng
    data_list = []
    filtered_labels = []
    
    for label, samples in data_table.items():
        if len(samples) > 1:
            # Lấy tối đa samples_per_class samples (random nếu có nhiều hơn)
            if len(samples) > samples_per_class:
                samples = random.sample(samples, samples_per_class)
            
            data_list.append(samples)
            filtered_labels.append(label)
    
    print(f"✓ Total labels: {len(data_table)}")
    print(f"✓ Labels with >1 sample: {len(data_list)}")
    print(f"✓ Samples per class (max): {samples_per_class}")
    print(f"✓ Total samples in filtered data: {sum(len(s) for s in data_list)}")
    
    return data_list, filtered_labels


# ============================================================================
# MODEL ARCHITECTURE (inspired by ResNet structure)
# ============================================================================
# ============================================================================
# MODEL ARCHITECTURE (DF-style)
# ============================================================================

class RawPacketEncoder(nn.Module):
    """
    DF-style Encoder for raw packet sequences
    Input: (batch, max_packets, 3) - [time, direction, size]
    Output: (batch, hidden_size)
    """
    def __init__(self, max_packets, hidden_size=256, embedding_size=128):
            super(RawPacketEncoder, self).__init__()
            self.max_packets = max_packets
            self.hidden_size = hidden_size
            self.embedding_size = embedding_size
    
            kernel_size = 8
            conv_stride = 1
            pool_stride = 4
            pool_size = 8
    
            self.conv1   = nn.Conv1d(3,   32,  kernel_size, stride=conv_stride)
            self.conv1_1 = nn.Conv1d(32,  32,  kernel_size, stride=conv_stride)
            self.conv2   = nn.Conv1d(32,  64,  kernel_size, stride=conv_stride)
            self.conv2_2 = nn.Conv1d(64,  64,  kernel_size, stride=conv_stride)
            self.conv3   = nn.Conv1d(64,  128, kernel_size, stride=conv_stride)
            self.conv3_3 = nn.Conv1d(128, 128, kernel_size, stride=conv_stride)
            self.conv4   = nn.Conv1d(128, 256, kernel_size, stride=conv_stride)
            self.conv4_4 = nn.Conv1d(256, 256, kernel_size, stride=conv_stride)
    
            self.batch_norm1 = nn.BatchNorm1d(32)
            self.batch_norm2 = nn.BatchNorm1d(64)
            self.batch_norm3 = nn.BatchNorm1d(128)
            self.batch_norm4 = nn.BatchNorm1d(256)
    
            self.max_pool_1 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
            self.max_pool_2 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
            self.max_pool_3 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
            self.max_pool_4 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
    
            self.dropout1 = nn.Dropout(p=0.1)
            self.dropout2 = nn.Dropout(p=0.1)
            self.dropout3 = nn.Dropout(p=0.1)
            self.dropout4 = nn.Dropout(p=0.1)
    
            # ⚠️ fc phải định nghĩa SAU khi tất cả conv/pool/dropout đã được tạo
            # vì _forward_convs dùng chúng
            with torch.no_grad():
                dummy = torch.zeros(1, 3, max_packets)
                dummy = self._forward_convs(dummy)
                flat_dim = dummy.view(1, -1).shape[1]
            print(f"  ✓ Flat dim after convs: {flat_dim}")
    
            self.fc = nn.Linear(flat_dim, hidden_size)
            self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _forward_convs(self, x):
        """Shared conv forward — dùng cho cả dummy pass và forward thật"""
        # ==== Block 1 (ELU) ====
        x = F.pad(x, (3, 4)); x = F.elu(self.conv1(x))
        x = F.pad(x, (3, 4)); x = F.elu(self.batch_norm1(self.conv1_1(x)))
        x = F.pad(x, (3, 4)); x = self.max_pool_1(x)
        x = self.dropout1(x)

        # ==== Block 2 (ReLU) ====
        x = F.pad(x, (3, 4)); x = F.relu(self.conv2(x))
        x = F.pad(x, (3, 4)); x = F.relu(self.batch_norm2(self.conv2_2(x)))
        x = F.pad(x, (3, 4)); x = self.max_pool_2(x)
        x = self.dropout2(x)

        # ==== Block 3 (ReLU) ====
        x = F.pad(x, (3, 4)); x = F.relu(self.conv3(x))
        x = F.pad(x, (3, 4)); x = F.relu(self.batch_norm3(self.conv3_3(x)))
        x = F.pad(x, (3, 4)); x = self.max_pool_3(x)
        x = self.dropout3(x)

        # ==== Block 4 (ReLU) ====
        x = F.pad(x, (3, 4)); x = F.relu(self.conv4(x))
        x = F.pad(x, (3, 4)); x = F.relu(self.batch_norm4(self.conv4_4(x)))
        x = F.pad(x, (3, 4)); x = self.max_pool_4(x)
        x = self.dropout4(x)

        return x

    def forward(self, x):
        # x: (batch, max_packets, 3)
        x = x.transpose(1, 2)      # -> (batch, 3, max_packets)
        x = self._forward_convs(x)
        x = x.view(x.size(0), -1)  # flatten
        feat = self.fc(x)           # -> (batch, hidden_size)
        return feat


class SupConPacketNet(nn.Module):
    """
    DF-style backbone + projection head for Supervised Contrastive Learning
    """
    def __init__(self, max_packets, hidden_size=256, embedding_size=128, head='mlp'):
        super(SupConPacketNet, self).__init__()
        self.max_packets = max_packets

        self.encoder = RawPacketEncoder(max_packets, hidden_size, embedding_size)

        # Projection head (DFsimCLR style)
        dim_mlp = hidden_size
        if head == 'linear':
            self.head = nn.Linear(dim_mlp, embedding_size)
        elif head == 'mlp':
            self.head = nn.Sequential(
                nn.Linear(dim_mlp, dim_mlp),
                nn.BatchNorm1d(dim_mlp),
                nn.ReLU(inplace=True),
                nn.Linear(dim_mlp, embedding_size)
            )
        else:
            raise NotImplementedError(f'head not supported: {head}')

    def forward(self, x):
        feat = self.encoder(x)
        feat = F.normalize(self.head(feat), dim=1)
        return feat

# class PacketConvBlock(nn.Module):
#     """Convolutional block for packet sequences"""
#     def __init__(self, in_channels, out_channels, kernel_size=5, stride=1):
#         super(PacketConvBlock, self).__init__()
#         padding = kernel_size // 2
        
#         self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, stride, padding, bias=False)
#         self.bn = nn.BatchNorm1d(out_channels)
#         self.relu = nn.ReLU(inplace=True)
    
#     def forward(self, x):
#         return self.relu(self.bn(self.conv(x)))


# class RawPacketEncoder(nn.Module):
#     """
#     Encoder for raw packet sequences
#     Input: (batch, max_packets, 3) - [time, direction, size]
#     Output: (batch, embedding_size)
#     """
#     def __init__(self, max_packets, hidden_size=256, embedding_size=128):
#         super(RawPacketEncoder, self).__init__()
#         self.max_packets = max_packets
#         self.hidden_size = hidden_size
#         self.embedding_size = embedding_size
        
#         # Feature extraction layers (similar to ResNet's conv layers)
#         self.conv1 = PacketConvBlock(3, 64, kernel_size=7, stride=1)
#         self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2)
        
#         self.conv2 = PacketConvBlock(64, 128, kernel_size=5, stride=1)
#         self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)
        
#         self.conv3 = PacketConvBlock(128, 256, kernel_size=3, stride=1)
#         self.conv4 = PacketConvBlock(256, 256, kernel_size=3, stride=1)
#         self.pool3 = nn.MaxPool1d(kernel_size=2, stride=2)
        
#         self.conv5 = PacketConvBlock(256, 512, kernel_size=3, stride=1)
#         self.conv6 = PacketConvBlock(512, 512, kernel_size=3, stride=1)
        
#         # Global pooling
#         self.avgpool = nn.AdaptiveAvgPool1d(1)
        
#         # Fully connected layers
#         self.fc = nn.Sequential(
#             nn.Linear(512, hidden_size),
#             nn.BatchNorm1d(hidden_size),
#             nn.ReLU(inplace=True),
#             nn.Dropout(0.3)
#         )
        
#         # Initialize weights
#         self._init_weights()
    
#     def _init_weights(self):
#         for m in self.modules():
#             if isinstance(m, nn.Conv1d):
#                 nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
#             elif isinstance(m, nn.BatchNorm1d):
#                 nn.init.constant_(m.weight, 1)
#                 nn.init.constant_(m.bias, 0)
#             elif isinstance(m, nn.Linear):
#                 nn.init.normal_(m.weight, 0, 0.01)
#                 if m.bias is not None:
#                     nn.init.constant_(m.bias, 0)
    
#     def forward(self, x):
#         # x: (batch, max_packets, 3)
#         x = x.transpose(1, 2)  # -> (batch, 3, max_packets)
        
#         # Feature extraction
#         x = self.conv1(x)
#         x = self.pool1(x)
        
#         x = self.conv2(x)
#         x = self.pool2(x)
        
#         x = self.conv3(x)
#         x = self.conv4(x)
#         x = self.pool3(x)
        
#         x = self.conv5(x)
#         x = self.conv6(x)
        
#         # Global pooling
#         x = self.avgpool(x)
#         x = x.squeeze(-1)
        
#         # Fully connected
#         feat = self.fc(x)
        
#         return feat


# class SupConPacketNet(nn.Module):
#     """
#     Supervised Contrastive Network for Packet Sequences
#     encoder + projection head (similar to SupConResNet)
#     """
#     def __init__(self, max_packets, hidden_size=256, embedding_size=128, head='mlp'):
#         super(SupConPacketNet, self).__init__()
#         self.max_packets = max_packets
        
#         self.encoder = RawPacketEncoder(max_packets, hidden_size, embedding_size)
        
#         # Projection head
#         if head == 'linear':
#             self.head = nn.Linear(hidden_size, embedding_size)
#         elif head == 'mlp':
#             self.head = nn.Sequential(
#                 nn.Linear(hidden_size, hidden_size),
#                 nn.ReLU(inplace=True),
#                 nn.Linear(hidden_size, embedding_size)
#             )
#         else:
#             raise NotImplementedError(f'head not supported: {head}')
    
#     def forward(self, x):
#         feat = self.encoder(x)
#         feat = F.normalize(self.head(feat), dim=1)
#         return feat


# ============================================================================
# DATA AUGMENTATION
# ============================================================================

def packet_augmentation(x):
    """
    Single augmentation function for packet sequences
    Apply random combination of augmentations
    """
    augmented = x.clone()
    
    # 1. Time jitter (randomly shift timestamps)
    if torch.rand(1).item() > 0.2:
        time_col = augmented[:, 0]
        jitter = (torch.rand_like(time_col) - 0.5) * 0.2  # ±10% jitter
        augmented[:, 0] = torch.clamp(time_col + jitter, min=0)
    
    # 2. Size scaling (scale packet sizes)
    if torch.rand(1).item() > 0.2:
        scale = 0.9 + torch.rand(1).item() * 0.2  # 0.9 to 1.1
        augmented[:, 2] = augmented[:, 2] * scale
    
    # 3. Packet dropout (randomly zero out some packets)
    if torch.rand(1).item() > 0.5:
        dropout_rate = 0.1
        mask = torch.rand(x.shape[0]) > dropout_rate
        mask = mask.unsqueeze(1).expand_as(x).to(x.device)
        augmented = augmented * mask.float()
    
    # 4. Gaussian noise
    if torch.rand(1).item() > 0.5:
        noise = torch.randn_like(augmented) * 0.05
        augmented = augmented + noise
    
    # Clean up NaN and Inf
    augmented = torch.nan_to_num(augmented, nan=0.0, posinf=0.0, neginf=0.0)
    
    return augmented


class TwoCropTransform:
    """Create two crops of the same sample (following original code structure)"""
    def __init__(self, transform):
        self.transform = transform
    
    def __call__(self, x):
        return [self.transform(x), self.transform(x)]


# ============================================================================
# DATASET
# ============================================================================

class PacketDataset(torch.utils.data.Dataset):
    """Dataset for packet sequences with two-view augmentation"""
    def __init__(self, data_list, transform=None):
        self.transform = transform
        self.samples = []
        self.labels = []
        
        for label_idx, samples in enumerate(data_list):
            for sample in samples:
                if isinstance(sample, np.ndarray):
                    sample = torch.from_numpy(sample.astype(np.float32))
                self.samples.append(sample)
                self.labels.append(label_idx)
        
        print(f"  Dataset: {len(self.samples)} samples, {len(data_list)} classes")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        label = self.labels[idx]
        
        if self.transform is not None:
            views = self.transform(sample)
            return views, label
        else:
            return sample, label


# ============================================================================
# TRAINING
# ============================================================================

def train(train_loader, model, criterion, optimizer, epoch):
    """One epoch training (following original code structure)"""
    model.train()
    
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    
    end = time.time()
    
    for idx, (images, labels) in enumerate(train_loader):
        data_time.update(time.time() - end)
        
        # images is a list of [view1, view2]
        images = torch.cat([images[0], images[1]], dim=0)
        
        if torch.cuda.is_available():
            images = images.cuda(non_blocking=True)
            labels = labels.cuda(non_blocking=True)
        
        bsz = labels.shape[0]
        
        # Warmup learning rate
        warmup_learning_rate(optimizer, epoch, idx, len(train_loader))
        
        # Forward
        features = model(images)
        f1, f2 = torch.split(features, [bsz, bsz], dim=0)
        features = torch.cat([f1.unsqueeze(1), f2.unsqueeze(1)], dim=1)
        
        # Compute loss
        loss = criterion(features, labels)
        
        # Check for NaN
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  ⚠️  WARNING: NaN/Inf loss at batch {idx}")
            continue
        
        # Update
        losses.update(loss.item(), bsz)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Measure time
        batch_time.update(time.time() - end)
        end = time.time()
        

    return losses.avg


# ============================================================================
# MAIN TRAINING PIPELINE
# ============================================================================

def main():
    """Main training function"""
    print("\n" + "="*60)
    print("RAW PACKET EMBEDDING WITH SUPCON")
    print("="*60)
    
    # Load data
    X, y, label_names, max_packets = load_and_process_raw_data(
        RAW_DATA_PATH, 
        LABEL_MAPPING_PATH
    )
    
    # Organize by label
    data_list, filtered_labels = organize_data_by_label(X, y)
    
    if len(data_list) < 2:
        print("\n❌ ERROR: Need at least 2 classes!")
        return False
    
    # Split train/test
    random.shuffle(data_list)
    n_train = max(1, int(len(data_list)))
    
    train_data_list = data_list[:n_train]
    test_data_list = data_list[n_train:] if n_train < len(data_list) else train_data_list
    
    print(f"\n✓ Data split:")
    print(f"  Training classes: {len(train_data_list)}")
    print(f"  Testing classes: {len(test_data_list)}")
    
    # Create datasets
    print(f"\n📦 Creating datasets...")
    transform = TwoCropTransform(packet_augmentation)
    train_dataset = PacketDataset(train_data_list, transform=transform)
    
    # Adjust batch size
    actual_batch_size = min(BATCH_SIZE, len(train_dataset))
    
    # Create dataloader
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=actual_batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        drop_last=(len(train_dataset) > actual_batch_size)
    )
    
    if len(train_loader) == 0:
        print("\n❌ ERROR: train_loader is empty!")
        return False
    
    print(f"✓ Train batches: {len(train_loader)}")
    
    # Build model
    print(f"\n🏗️  Building model...")
    model = SupConPacketNet(
        max_packets=max_packets,
        hidden_size=HIDDEN_SIZE,
        embedding_size=EMBEDDING_SIZE,
        head='mlp'
    )
    criterion = SupConLoss(temperature=TEMPERATURE)
    
    if torch.cuda.is_available():
        model = model.cuda()
        criterion = criterion.cuda()
        print("✓ Using CUDA")
    else:
        print("✓ Using CPU")
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    print(f"✓ Model parameters: {total_params:,}")
    
    # Optimizer (following original code)
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY
    )
    
    print(f"\n🚀 Starting training...")
    print(f"  Epochs: {EPOCHS}")
    print(f"  Batch size: {actual_batch_size}")
    print(f"  Learning rate: {LEARNING_RATE}")
    print(f"  Temperature: {TEMPERATURE}")
    print(f"  Warmup: {WARMUP_EPOCHS} epochs" if WARMUP else "  No warmup")
    print(f"  LR schedule: {'Cosine' if COSINE_SCHEDULE else 'Step'}")
    
    # Create save directory
    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
    
    # Training loop
    for epoch in range(1, EPOCHS + 1):
        # Adjust learning rate
        adjust_learning_rate(optimizer, epoch)
        
        # Train
        time1 = time.time()
        loss = train(train_loader, model, criterion, optimizer, epoch)
        time2 = time.time()
        if epoch % 50 == 0:
            print(f'epoch {epoch}, total time {time2 - time1:.2f}, loss {loss}' )
        
    
    # Save final model
    save_model(model, optimizer, EPOCHS, MODEL_PATH)
    
    print(f"\n{'='*60}")
    print("✅ TRAINING COMPLETED")
    print(f"{'='*60}")
    print(f"Model saved to: {MODEL_PATH}")
    
    return True


if __name__ == '__main__':
    success = main()
    
    if not success:
        print("\n❌ Training failed")
        sys.exit(1)

In [ ]:
# ============================================================================
# SUBSAMPLING
# ============================================================================

def subsample_per_class(X, y, n_per_class=100, seed=42):
    """Lấy tối đa n_per_class samples cho mỗi class để giảm tải khi train."""
    rng = np.random.RandomState(seed)
    selected_idx = []

    for label in np.unique(y):
        idx = np.where(y == label)[0]
        if len(idx) > n_per_class:
            idx = rng.choice(idx, size=n_per_class, replace=False)
        selected_idx.append(idx)

    selected_idx = np.concatenate(selected_idx)
    rng.shuffle(selected_idx)

    return X[selected_idx], y[selected_idx]

In [ ]:
import numpy as np
import pandas as pd
import os
import torch
from torch import nn
import torch.nn.functional as F

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# ============================================================================
# CONFIGURATION
# ============================================================================
IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    EMBEDDING_DATA_PATH = '/kaggle/input/datasets/linhhphng/100-topsearch/100-topsearch/icloud_100/train_data.csv'
    TEST_DATA_PATH = '/kaggle/input/datasets/linhhphng/100-topsearch/100-topsearch/icloud_100/test_data.csv'
    LABEL_MAPPING_PATH = '/kaggle/input/datasets/linhhphng/100-topsearch/100-topsearch/icloud_100/label_mapping.csv'
    MODEL_PATH = '/kaggle/working/pretrain_topsearch10.pth'
    OUTPUT_FOLDER = '/kaggle/working/output'

def load_pretrained_embedding_model(model_path):
    print(f"\n📥 Loading pretrained embedding model from: {model_path}")
    
    if not os.path.exists(model_path):
        print(f"  ❌ Model file not found!")
        return None
    
    checkpoint = torch.load(model_path, map_location='cpu')
    config = checkpoint.get('model_config', {})
    max_packets = config.get('max_packets', 25000)
    hidden_size = config.get('hidden_size', 256)
    embedding_size = config.get('embedding_size', 128)
    
    model = SupConPacketNet(max_packets=max_packets, hidden_size=hidden_size, 
                           embedding_size=embedding_size, head='mlp')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    
    print(f"  ✓ Model loaded on {device} (epoch {checkpoint.get('epoch', '?')})")
    return model

# code cũ sau PH
# def extract_embeddings_from_raw(model, X_raw, batch_size=32):
#     model.eval()
#     device = next(model.parameters()).device
#     embeddings = []
    
#     with torch.no_grad():
#         for i in range(0, len(X_raw), batch_size):
#             batch = X_raw[i:i + batch_size]
#             batch_tensor = torch.FloatTensor(batch).to(device)
#             batch_emb = model(batch_tensor)
#             embeddings.append(batch_emb.cpu().numpy())
    
#     return np.vstack(embeddings)


# code cũ trước PH
# def extract_embeddings_from_raw(model, X_raw, batch_size=32):
#     """Extract 512-dim features from the CNN backbone, before the fc and projection head."""
#     encoder = model.encoder
#     encoder.eval()
#     device = next(model.parameters()).device
#     embeddings = []

#     with torch.no_grad():
#         for i in range(0, len(X_raw), batch_size):
#             batch = X_raw[i:i + batch_size]
#             batch_tensor = torch.FloatTensor(batch).to(device)

#             # Replicate RawPacketEncoder.forward() but stop after avgpool (512-dim)
#             x = batch_tensor.transpose(1, 2)
#             x = encoder.conv1(x)
#             x = encoder.pool1(x)
#             x = encoder.conv2(x)
#             x = encoder.pool2(x)
#             x = encoder.conv3(x)
#             x = encoder.conv4(x)
#             x = encoder.pool3(x)
#             x = encoder.conv5(x)
#             x = encoder.conv6(x)
#             x = encoder.avgpool(x)
#             x = x.squeeze(-1)          # shape: (batch, 512)

#             embeddings.append(x.cpu().numpy())

#     return np.vstack(embeddings)

# df model trước ph
def extract_embeddings_from_raw(model, X_raw, batch_size=32):
    """Extract features after encoder fc, before projection head — shape: (batch, hidden_size)"""
    encoder = model.encoder
    encoder.eval()
    device = next(model.parameters()).device
    embeddings = []

    with torch.no_grad():
        for i in range(0, len(X_raw), batch_size):
            batch = X_raw[i:i + batch_size]
            batch_tensor = torch.FloatTensor(batch).to(device)
            
            # encoder.forward() = _forward_convs + flatten + fc -> (batch, hidden_size)
            feat = encoder(batch_tensor)
            embeddings.append(feat.cpu().numpy())

    return np.vstack(embeddings)

# ============================================================================
# DATA LOADING
# ============================================================================

def load_raw_packet_data(data_path, label_mapping_path, data_type='data', 
                          n_per_class=None, chunksize=2000, seed=42):
    """
    n_per_class: nếu set, chỉ giữ tối đa n_per_class samples/class, đọc theo chunk
    để tránh load hết file vào RAM.
    """
    print(f"\n{'='*60}")
    print(f"LOADING {data_type.upper()} DATA")
    print(f"{'='*60}")
    
    if os.path.exists(label_mapping_path):
        label_df = pd.read_csv(label_mapping_path)
        label_names = dict(zip(label_df['label_name'], label_df['label_id']))
        print(f"✓ Loaded {len(label_names)} label mappings")
    else:
        label_names = {}
    
    if not os.path.exists(data_path):
        print(f"❌ Data file not found: {data_path}")
        return None, None, label_names
    
    rng = np.random.RandomState(seed)
    
    all_samples = []
    all_labels = []
    class_counts = {}  # label -> số sample đã giữ
    total_rows_seen = 0
    
    for chunk in pd.read_csv(data_path, header=None, chunksize=chunksize):
        total_rows_seen += len(chunk)
        total_cols = len(chunk.columns)
        max_packets = total_cols // 4
        
        for _, row in chunk.iterrows():
            sample = row.values.reshape(max_packets, 4)
            label = int(sample[0, 0])
            
            if n_per_class is not None:
                count = class_counts.get(label, 0)
                if count >= n_per_class:
                    continue  # đủ quota cho class này rồi, bỏ qua luôn -> nhẹ RAM
                class_counts[label] = count + 1
            
            sample_features = sample[:, 1:]
            all_samples.append(sample_features)
            all_labels.append(label)
        
        print(f"  ...đã quét {total_rows_seen} dòng, đang giữ {len(all_samples)} samples", end='\r')
    
    print()  # xuống dòng sau progress
    
    X = np.array(all_samples, dtype=np.float32)
    y = np.array(all_labels, dtype=np.int64)
    
    print(f"✓ X: {X.shape}, y: {y.shape}, labels: {len(np.unique(y))}")
    return X, y, label_names
    
# ============================================================================
# MAIN
# ============================================================================

def main():
    print("\n" + "="*60)
    print("RANDOM FOREST CLASSIFIER WITH EMBEDDINGS")
    print("="*60)
    
    # Load model
    emb_model = load_pretrained_embedding_model(MODEL_PATH)
    if emb_model is None:
        return
    
    # # Load data
    # X_train_raw, y_train, label_names = load_raw_packet_data(EMBEDDING_DATA_PATH, LABEL_MAPPING_PATH, 'TRAIN')
    # X_test_raw, y_test, _ = load_raw_packet_data(TEST_DATA_PATH, LABEL_MAPPING_PATH, 'TEST')
    
    # if X_train_raw is None or X_test_raw is None:
    #     return
    
    # # Extract embeddings
    # print("\n🔄 Extracting embeddings...")
    # X_train_emb = extract_embeddings_from_raw(emb_model, X_train_raw)
    # X_test_emb = extract_embeddings_from_raw(emb_model, X_test_raw)
    
    # print(f"  ✓ Train embeddings: {X_train_emb.shape}")
    # print(f"  ✓ Test embeddings: {X_test_emb.shape}")

    # Load data
    # Load data — train chỉ lấy 100 samples/class, đọc theo chunk để nhẹ RAM
    X_train_raw, y_train, label_names = load_raw_packet_data(
        EMBEDDING_DATA_PATH, LABEL_MAPPING_PATH, 'TRAIN', n_per_class=200
    )
    X_test_raw, y_test, _ = load_raw_packet_data(
        TEST_DATA_PATH, LABEL_MAPPING_PATH, 'TEST'
    )
    
    if X_train_raw is None or X_test_raw is None:
        return

    # Giảm số lượng train samples xuống 100/class để load nhẹ hơn
    print("\n✂️  Subsampling train set to 100 samples/class...")
    print(f"  Before: {X_train_raw.shape}")
    X_train_raw, y_train = subsample_per_class(X_train_raw, y_train, n_per_class=100, seed=42)
    print(f"  After:  {X_train_raw.shape}")
    
    # Extract embeddings
    print("\n🔄 Extracting embeddings...")
    X_train_emb = extract_embeddings_from_raw(emb_model, X_train_raw)
    X_test_emb = extract_embeddings_from_raw(emb_model, X_test_raw)
    
    # Train Random Forest
    print(f"\n{'='*60}")
    print("TRAINING RANDOM FOREST")
    print(f"{'='*60}")
    
    clf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42, verbose=1)
    print("Training...")
    clf.fit(X_train_emb, y_train)
    
    # Predict
    print("\nPredicting on test set...")
    y_pred = clf.predict(X_test_emb)
    
    # Get label names for classification report
    unique_labels = np.unique(np.concatenate([y_train, y_test]))
    target_names = [label_names.get(label, f'Class_{label}') for label in unique_labels]
    
    # Print classification report
    print(f"\n{'='*60}")
    print("CLASSIFICATION REPORT")
    print(f"{'='*60}\n")
    
    report = classification_report(
        y_test,
        y_pred,
        target_names=[str(x) for x in target_names],
        digits=4,
        zero_division=0
    )
    print(report)
    
    # Print confusion matrix info
    print(f"\n{'='*60}")
    print("CONFUSION MATRIX SUMMARY")
    print(f"{'='*60}\n")
    
    cm = confusion_matrix(y_test, y_pred, labels=unique_labels)
    
    for i, label in enumerate(unique_labels):
        label_name = label_names.get(label, f'Class_{label}')
        correct = cm[i, i]
        total = cm[i, :].sum()
        accuracy = correct / total if total > 0 else 0
        print(f"{label_name:30s} - Correct: {correct:5d}/{total:5d} ({accuracy:.2%})")
    
    # Save detailed results
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    
    # Save classification report to file
    report_path = os.path.join(OUTPUT_FOLDER, 'classification_report.txt')
    with open(report_path, 'w') as f:
        f.write("="*60 + "\n")
        f.write("CLASSIFICATION REPORT - RANDOM FOREST\n")
        f.write("="*60 + "\n\n")
        f.write(report)
    
    print(f"\n✓ Classification report saved to: {report_path}")

if __name__ == '__main__':
    main()